In [ ]:
"""
XGBoost Kaggle submission — Small Reptile Species Distribution (EcoStat Modelling).

Trains on the FULL train.csv (all labelled data) and predicts presence probabilities
for the Kaggle test.csv, writing a submission file in the required `id,pred` format.

This is the SUBMISSION pipeline, distinct from the internal-evaluation model:
  * It uses all of train.csv, not the 80/20 split (more data -> better final model).
  * It predicts on the real test.csv, which has no pres.abs column.

Calibration note (important for this competition):
  The Kaggle metric is log-score (log loss), which rewards well-calibrated probabilities.
  scale_pos_weight (used in the evaluation model to help recall/AUC) biases probabilities
  upward and tends to WORSEN log loss, so it is OFF here by default. Set
  USE_SCALE_POS_WEIGHT = True only if you want to experiment.

How n_estimators is chosen:
  5-fold stratified CV with early stopping estimates the best number of trees and the
  expected log loss. The final model is then refit on ALL of train.csv using that tree
  count, so no labelled data is wasted on a hold-out for the submission.

Usage:
    python xgboost_kaggle_submit.py --train train.csv --test test.csv --out submission.csv
"""

In [1]:
import argparse
import warnings
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore", category=UserWarning)

# ----------------------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------------------
TARGET = "pres.abs"
SPECIES = "Species"
ID_COL = "id"
NUMERIC_FEATURES = ["disturb", "rainann", "soildepth", "soilfert",
                    "tempann", "topo", "easting", "northing"]
RANDOM_STATE = 42
N_SPLITS = 5
MAX_ROUNDS = 3000
EARLY_STOP = 50
CLIP_EPS = 1e-6              # keep probabilities strictly inside (0, 1) for the log-score
USE_SCALE_POS_WEIGHT = False  # see calibration note above

# Model hyperparameters. Replace with the winners from xgboost_tune.py if you have them.
PARAMS = dict(
    learning_rate=0.03,
    max_depth=4,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=2.0,
    gamma=0.0,
)


# ----------------------------------------------------------------------------------
# Feature building (consistent with the other scripts)
# ----------------------------------------------------------------------------------
def build_X(df, columns=None):
    species_dummies = pd.get_dummies(df[SPECIES], prefix="sp")
    numeric_present = [c for c in NUMERIC_FEATURES if c in df.columns]
    X = pd.concat([df[numeric_present].reset_index(drop=True),
                   species_dummies.reset_index(drop=True)], axis=1)
    if columns is not None:                       # align test -> train columns
        X = X.reindex(columns=columns, fill_value=0)
    return X


def make_model(n_estimators, scale_pos_weight, early_stopping):
    return xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=n_estimators,
        scale_pos_weight=scale_pos_weight,
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        early_stopping_rounds=EARLY_STOP if early_stopping else None,
        **PARAMS,
    )


def estimate_rounds_and_score(X, y):
    """5-fold stratified CV: returns mean best iteration and mean CV log loss."""
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    best_iters, losses = [], []
    for tr_idx, val_idx in skf.split(X, y):
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
        spw = ((len(y_tr) - int(y_tr.sum())) / max(int(y_tr.sum()), 1)
               if USE_SCALE_POS_WEIGHT else 1.0)
        m = make_model(MAX_ROUNDS, spw, early_stopping=True)
        m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        p = np.clip(m.predict_proba(X_val)[:, 1], CLIP_EPS, 1 - CLIP_EPS)
        losses.append(log_loss(y_val, p, labels=[0, 1]))
        best_iters.append(m.best_iteration + 1)
    return int(np.mean(best_iters)), float(np.mean(losses)), float(np.std(losses))


def main():
    parser = argparse.ArgumentParser(description="XGBoost Kaggle submission builder.")
    parser.add_argument("--train", default="train.csv")
    parser.add_argument("--test", default="test.csv")
    parser.add_argument("--out", default="submission.csv")
    args, _ = parser.parse_known_args()  # notebook-safe

    train_df = pd.read_csv(args.train)
    test_df = pd.read_csv(args.test)
    print(f"train: {len(train_df)} rows (presence {train_df[TARGET].mean():.4f}) | "
          f"test: {len(test_df)} rows")

    # Features: fit columns on train, align test to them.
    X = build_X(train_df)
    y = train_df[TARGET].astype(int).reset_index(drop=True)
    feature_cols = list(X.columns)
    X_test = build_X(test_df, columns=feature_cols)

    # 1) Estimate tree count + expected Kaggle log loss by CV.
    print("\nEstimating n_estimators and expected log loss (5-fold CV) ...")
    n_best, cv_mean, cv_std = estimate_rounds_and_score(X, y)
    print(f"  mean best iteration : {n_best}")
    print(f"  expected log loss   : {cv_mean:.4f} (+/- {cv_std:.4f})")

    # 2) Refit on ALL training data with that tree count (no early stopping).
    print("\nRefitting on the full training set ...")
    spw = ((len(y) - int(y.sum())) / max(int(y.sum()), 1)
           if USE_SCALE_POS_WEIGHT else 1.0)
    model = make_model(n_best, spw, early_stopping=False)
    model.fit(X, y, verbose=False)

    # 3) Predict probabilities for the Kaggle test set.
    pred = np.clip(model.predict_proba(X_test)[:, 1], CLIP_EPS, 1 - CLIP_EPS)

    # 4) Write submission in the required id,pred format.
    if ID_COL not in test_df.columns:
        raise ValueError(f"'{ID_COL}' column missing from {args.test}; needed for submission.")
    submission = pd.DataFrame({ID_COL: test_df[ID_COL], "pred": pred})
    submission.to_csv(args.out, index=False)

    print(f"\nWrote {args.out} with {len(submission)} rows.")
    print(f"  prediction range: [{pred.min():.4f}, {pred.max():.4f}], "
          f"mean {pred.mean():.4f}")
    print(submission.head().to_string(index=False))

    model.save_model("xgboost_kaggle_model.json")
    print("\nSaved fitted model to xgboost_kaggle_model.json")


if __name__ == "__main__":
    main()

train: 5128 rows (presence 0.0634) | test: 2936 rows

Estimating n_estimators and expected log loss (5-fold CV) ...
  mean best iteration : 314
  expected log loss   : 0.1515 (+/- 0.0083)

Refitting on the full training set ...

Wrote submission.csv with 2936 rows.
  prediction range: [0.0018, 0.6981], mean 0.0501
  id     pred
5129 0.009075
5130 0.038900
5131 0.009673
5132 0.005895
5133 0.003062

Saved fitted model to xgboost_kaggle_model.json
